## Parameters

In [0]:
# NYC Yellow Taxi — Incremental Gold Layer

dbutils.widgets.text("year", "2026")
dbutils.widgets.text("month", "07")

year = dbutils.widgets.get("year")
month = dbutils.widgets.get("month")

print(f"Incremental Gold processing: {year}-{month}")

In [0]:
# Silver Input

from pyspark.sql import functions as F

silver_path = (
    f"abfss://silver@stnyctaxide2026nandi.dfs.core.windows.net/"
    f"yellow_taxi/{year}/{month}"
)

try:
    df_silver = spark.read.parquet(silver_path)

    silver_count = df_silver.count()

    print(f"Silver path: {silver_path}")
    print(f"Silver records: {silver_count:,}")
except Exception as e:
    print(f"Silver data is not available for {year}-{month}.")
    print(f"Expected path: {silver_path}")
    print("Incremental Gold processing will wait for this month to arrive.")


In [0]:
year = "2026"
month = "07"

print(f"Testing incremental Gold logic: {year}-{month}")

In [0]:
from pyspark.sql import functions as F

silver_path = (
    f"abfss://silver@stnyctaxide2026nandi.dfs.core.windows.net/"
    f"yellow_taxi/{year}/{month}"
)

df_silver = spark.read.parquet(silver_path)

print(f"Silver path: {silver_path}")
print(f"Silver records: {df_silver.count():,}")

## sample

In [0]:
df_silver.limit(10000)

In [0]:
df_sample = df_silver.limit(10000)

print(f"Sample records: {df_sample.count():,}")

## Read the persisted SCD2 dimension

In [0]:
gold_zone_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "dimensions/taxi_zone"
)

df_zone = (
    spark.read
    .format("delta")
    .load(gold_zone_path)
)

print(f"Taxi zone dimension rows: {df_zone.count():,}")

## Prepare zone lookups

In [0]:
pickup_zone = (
    df_zone
    .filter(F.col("is_current") == True)
    .select(
        F.col("zone_sk").alias("pickup_zone_sk"),
        F.col("location_id").alias("pickup_location_id")
    )
)

dropoff_zone = (
    df_zone
    .filter(F.col("is_current") == True)
    .select(
        F.col("zone_sk").alias("dropoff_zone_sk"),
        F.col("location_id").alias("dropoff_location_id")
    )
)

print(f"Current pickup zones: {pickup_zone.count():,}")
print(f"Current drop-off zones: {dropoff_zone.count():,}")

## Join the sample

In [0]:
df_test = (
    df_sample
    .join(pickup_zone, "pickup_location_id", "left")
    .join(dropoff_zone, "dropoff_location_id", "left")
)

print(f"Test rows after joins: {df_test.count():,}")

## Check unresolved zone keys

In [0]:
df_test.select(
    F.sum(
        F.when(F.col("pickup_zone_sk").isNull(), 1).otherwise(0)
    ).alias("unresolved_pickup_zones"),
    F.sum(
        F.when(F.col("dropoff_zone_sk").isNull(), 1).otherwise(0)
    ).alias("unresolved_dropoff_zones")
).show()

## Create incremental Gold fact sample

In [0]:
df_fact_test = (
    df_test
    .select(
        "pickup_date",
        "pickup_datetime",
        "dropoff_datetime",
        "vendor_id",
        "pickup_zone_sk",
        "dropoff_zone_sk",
        "passenger_count",
        "trip_distance",
        "trip_duration_minutes",
        "trip_speed_mph",
        "rate_code_id",
        "payment_type",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "congestion_surcharge",
        "airport_fee",
        "cbd_congestion_fee",
        "total_amount",
        "source_year",
        "source_month",
        "source_file",
        "ingestion_timestamp"
    )
)

print(f"Incremental Gold fact sample: {df_fact_test.count():,}")

## validate the fact quality

In [0]:
df_fact_test.select(
    F.count("*").alias("total_rows"),
    F.sum(
        F.when(F.col("pickup_zone_sk").isNull(), 1).otherwise(0)
    ).alias("null_pickup_zone_sk"),
    F.sum(
        F.when(F.col("dropoff_zone_sk").isNull(), 1).otherwise(0)
    ).alias("null_dropoff_zone_sk"),
    F.sum(
        F.when(F.col("pickup_datetime").isNull(), 1).otherwise(0)
    ).alias("null_pickup_datetime"),
    F.sum(
        F.when(F.col("total_amount").isNull(), 1).otherwise(0)
    ).alias("null_total_amount"),
    F.sum(
        F.when(F.col("trip_distance") < 0, 1).otherwise(0)
    ).alias("negative_distance"),
    F.sum(
        F.when(F.col("total_amount") < 0, 1).otherwise(0)
    ).alias("negative_total_amount")
).show()

## production-safe incremental write

In [0]:
gold_fact_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "facts/taxi_trip"
)

df_gold_existing = (
    spark.read
    .format("delta")
    .load(gold_fact_path)
)

existing_month = (
    df_gold_existing
    .filter(
        (F.col("source_year") == int(year)) &
        (F.col("source_month") == int(month))
    )
    .count()
)

print(f"Existing Gold rows for {year}-{month}: {existing_month:,}")

In [0]:
df_gold_july = (
    df_gold_existing
    .filter(
        (F.col("source_year") == 2026) &
        (F.col("source_month") == 7)
    )
)

print("Silver July:", df_silver.count())
print("Gold July:", df_gold_july.count())

print("Silver distinct source files:")
df_silver.select("source_file").distinct().show(truncate=False)

print("Gold distinct source files:")
df_gold_july.select("source_file").distinct().show(truncate=False)

## reconcile Silver vs Gold

In [0]:
silver_keys = (
    df_silver
    .select(
        "pickup_datetime",
        "dropoff_datetime",
        "vendor_id",
        "total_amount",
        "trip_distance"
    )
    .dropDuplicates()
)

gold_keys = (
    df_gold_july
    .select(
        "pickup_datetime",
        "dropoff_datetime",
        "vendor_id",
        "total_amount",
        "trip_distance"
    )
    .dropDuplicates()
)

missing_in_gold = (
    silver_keys
    .join(
        gold_keys,
        [
            "pickup_datetime",
            "dropoff_datetime",
            "vendor_id",
            "total_amount",
            "trip_distance"
        ],
        "left_anti"
    )
)

print(
    f"Rows in Silver not found in Gold: "
    f"{missing_in_gold.count():,}"
)

missing_in_gold.show(truncate=False)

# Incremental Gold Write Control

In [0]:
# Check whether this source period is already present in Gold

existing_month_rows = (
    df_gold_existing
    .filter(
        (F.col("source_year") == int(year)) &
        (F.col("source_month") == int(month))
    )
    .count()
)

if existing_month_rows > 0:
    print(
        f"Source period {year}-{month} already exists in Gold "
        f"with {existing_month_rows:,} rows."
    )
    print("Incremental write will be skipped to prevent duplicate data.")
else:
    print(
        f"Source period {year}-{month} is not present in Gold."
    )
    print("Incremental write is eligible.")

## create the incremental Gold fact

In [0]:
df_fact_incremental = (
    df_test
    .select(
        "pickup_date",
        "pickup_datetime",
        "dropoff_datetime",
        "vendor_id",
        "pickup_zone_sk",
        "dropoff_zone_sk",
        "passenger_count",
        "trip_distance",
        "trip_duration_minutes",
        "trip_speed_mph",
        "rate_code_id",
        "payment_type",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "congestion_surcharge",
        "airport_fee",
        "cbd_congestion_fee",
        "total_amount",
        "source_year",
        "source_month",
        "source_file",
        "ingestion_timestamp"
    )
)

print(f"Incremental Gold sample rows: {df_fact_incremental.count():,}")

In [0]:
# validate 

df_fact_incremental.select(
    F.count("*").alias("total_rows"),
    F.sum(
        F.when(F.col("pickup_zone_sk").isNull(), 1).otherwise(0)
    ).alias("null_pickup_zone_sk"),
    F.sum(
        F.when(F.col("dropoff_zone_sk").isNull(), 1).otherwise(0)
    ).alias("null_dropoff_zone_sk"),
    F.sum(
        F.when(F.col("pickup_datetime").isNull(), 1).otherwise(0)
    ).alias("null_pickup_datetime"),
    F.sum(
        F.when(F.col("total_amount").isNull(), 1).otherwise(0)
    ).alias("null_total_amount"),
    F.sum(
        F.when(F.col("trip_distance") < 0, 1).otherwise(0)
    ).alias("negative_distance"),
    F.sum(
        F.when(F.col("total_amount") < 0, 1).otherwise(0)
    ).alias("negative_total_amount")
).show()

In [0]:
df_fact_incremental.groupBy(
    "source_year",
    "source_month",
    "source_file"
).count().show(truncate=False)

### duplicate/idempotency validation

In [0]:
existing_keys = (
    df_gold_existing
    .select(
        "pickup_datetime",
        "dropoff_datetime",
        "vendor_id",
        "total_amount",
        "source_year",
        "source_month"
    )
    .dropDuplicates()
)

incremental_keys = (
    df_fact_incremental
    .select(
        "pickup_datetime",
        "dropoff_datetime",
        "vendor_id",
        "total_amount",
        "source_year",
        "source_month"
    )
)

duplicate_count = (
    incremental_keys
    .join(
        existing_keys,
        [
            "pickup_datetime",
            "dropoff_datetime",
            "vendor_id",
            "total_amount",
            "source_year",
            "source_month"
        ],
        "inner"
    )
    .count()
)

print(f"Existing matching rows: {duplicate_count:,}")

## Replace the test sample with production-ready monthly logic

In [0]:
year = "2026"
month = "07"

print(f"Incremental Gold processing: {year}-{month}")

In [0]:
df_month = df_silver

In [0]:
pickup_zone = (
    df_zone
    .filter(F.col("is_current") == True)
    .select(
        F.col("zone_sk").alias("pickup_zone_sk"),
        F.col("location_id").alias("pickup_location_id")
    )
)

dropoff_zone = (
    df_zone
    .filter(F.col("is_current") == True)
    .select(
        F.col("zone_sk").alias("dropoff_zone_sk"),
        F.col("location_id").alias("dropoff_location_id")
    )
)

## full monthly Gold fact transformation

In [0]:
df_fact_incremental = (
    df_month
    .join(pickup_zone, "pickup_location_id", "left")
    .join(dropoff_zone, "dropoff_location_id", "left")
    .select(
        "pickup_date",
        "pickup_datetime",
        "dropoff_datetime",
        "vendor_id",
        "pickup_zone_sk",
        "dropoff_zone_sk",
        "passenger_count",
        "trip_distance",
        "trip_duration_minutes",
        "trip_speed_mph",
        "rate_code_id",
        "payment_type",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "congestion_surcharge",
        "airport_fee",
        "cbd_congestion_fee",
        "total_amount",
        "source_year",
        "source_month",
        "source_file",
        "ingestion_timestamp"
    )
)

In [0]:
df_fact_incremental = (
    df_fact_incremental
    .filter(F.col("dq_reason").isNull())
)

## Add the production idempotency guard

In [0]:
existing_month_rows = (
    df_gold_existing
    .filter(
        (F.col("source_year") == int(year)) &
        (F.col("source_month") == int(month))
    )
    .count()
)

if existing_month_rows > 0:
    print(
        f"Source period {year}-{month} already exists in Gold "
        f"with {existing_month_rows:,} rows."
    )
    print("Incremental write will be skipped.")
    incremental_write_allowed = False
else:
    print(
        f"Source period {year}-{month} is not present in Gold."
    )
    print("Incremental write is allowed.")
    incremental_write_allowed = True

## the actual write block

In [0]:
if incremental_write_allowed:

    print(f"Starting Gold write for source period {year}-{month}.")

    (
        df_fact_incremental
        .write
        .format("delta")
        .mode("append")
        .partitionBy("pickup_date")
        .save(gold_fact_path)
    )

    print(
        f"Gold incremental write completed for source period "
        f"{year}-{month}."
    )

else:

    print(
        f"Source period {year}-{month} already exists in Gold."
    )
    print("Gold write skipped to maintain idempotency.")

In [0]:
df_month = (
    df_silver
    .filter(F.col("dq_reason").isNull())
)

### add a safety check for unresolved zones

In [0]:
df_fact_incremental = df_fact_incremental.filter(
    F.col("pickup_zone_sk").isNotNull() &
    F.col("dropoff_zone_sk").isNotNull()
)